# OpenCV DNN and ONNX Inference

> **Advanced · Deep learning inference**


## Why this matters

OpenCV can run exported neural networks, but correct preprocessing and output parsing are part of the model contract. ONNX makes that contract portable, not automatic.

**Where it appears:** Portable model inference, image classification/detection prototypes, model inspection, and integration with traditional preprocessing.


## Learning Objectives

- Understand the DNN module's blob-based inference API
- Correctly configure preprocessing (mean subtraction, scaling, channel order) per model
- Parse raw network output into usable detections
- Understand ONNX as a framework-agnostic model interchange format
- Load and run ONNX models via cv2.dnn.readNetFromONNX
- Inspect model input/output shapes before writing inference code around them


## Prerequisites

02 NumPy for Images; 03 OpenCV Setup and Your First Pipeline

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.dnn.blobFromImage`, `readNetFromONNX`, `setInput`, `forward`, ONNX model inspection

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### OpenCV DNN Module Fundamentals

`cv2.dnn` runs pretrained neural networks (TensorFlow, TensorFlow, ONNX, Darknet
formats) directly in OpenCV without needing the original training
framework installed. The critical, easy-to-get-wrong step is
**preprocessing**: `cv2.dnn.blobFromImage` needs the correct scale factor,
mean-subtraction values, target size, and channel-swap flag -- these must
exactly match what the specific model was trained with, or inference
silently produces garbage (no error is raised for wrong preprocessing).


### Working with ONNX Models

ONNX (Open Neural Network Exchange) lets a model trained in PyTorch,
TensorFlow, or another framework be exported once and run anywhere,
including in `cv2.dnn` with no dependency on the original training
framework. Before writing any inference code, a model's actual input
shape, output shape, and expected preprocessing must be inspected --
assuming a shape (e.g. always 224x224) causes silent shape-mismatch
errors or garbage output for models trained differently.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: OpenCV DNN Module Fundamentals


### 1. The blobFromImage preprocessing contract

Build a small reference table of preprocessing parameters for common model families, and a function that documents/applies them explicitly rather than guessing values.



In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid

# Reference preprocessing recipes for common pretrained model families.
# Mismatching these for a given model is the #1 cause of "the model works but gives garbage".
PREPROCESS_RECIPES = {
    "caffe_vgg_style": dict(
        scalefactor=1.0, size=(224, 224), mean=(104, 117, 123), swapRB=False
    ),
    "tensorflow_mobilenet": dict(
        scalefactor=1 / 127.5, size=(300, 300), mean=(127.5, 127.5, 127.5), swapRB=True
    ),
    "darknet_yolo": dict(
        scalefactor=1 / 255.0, size=(416, 416), mean=(0, 0, 0), swapRB=True
    ),
}


def make_blob(image: np.ndarray, recipe_name: str) -> np.ndarray:
    recipe = PREPROCESS_RECIPES[recipe_name]
    return cv2.dnn.blobFromImage(
        image, recipe["scalefactor"], recipe["size"], recipe["mean"], recipe["swapRB"]
    )


demo_image = load_real_image("images/standard", "dog.jpg")
for name in PREPROCESS_RECIPES:
    blob = make_blob(demo_image, name)
    print(f"{name:22s} -> blob shape {blob.shape}")

### 2. Running inference with a bundled classical net (edge detection as a 'DNN')

To demonstrate the full `cv2.dnn` forward-pass mechanics without requiring a large downloaded model, run OpenCV's bundled HED (Holistically-Nested Edge Detection) style API pattern using a stand-in: here we show the *mechanics* using a Sobel-based function wrapped in the same 'blob in, map out' shape a real DNN uses.



In [ ]:
def run_forward_pass(
    net: "cv2.dnn.Net", blob: np.ndarray, output_layer: str | None = None
):
    """Standard OpenCV DNN forward-pass pattern: setInput -> forward -> raw output tensor."""
    net.setInput(blob)
    return net.forward(output_layer) if output_layer else net.forward()


# Load the real Caffe DNN model we downloaded earlier
proto_path = get_real_data("models", "opencv_face_detector.pbtxt")
model_path = get_real_data("models", "opencv_face_detector_uint8.pb")
image = load_real_image("images/faces", "face.jpg")

# Load the real network
net = cv2.dnn.readNetFromTensorflow(str(model_path), str(proto_path))

# Preprocess image into a blob suitable for the network (SSD expects 300x300 and TensorFlow-style mean subtraction)
blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0), swapRB=False)

# Run the real forward pass
output = run_forward_pass(net, blob)

print("Real inference successful!")
print("Input image shape:   ", image.shape)
print("Blob shape:          ", blob.shape)
print("Output tensor shape: ", output.shape)

### 3. Parsing typical detection-network output

Most OpenCV DNN detection models (SSD/TensorFlow-style) return a fixed-shape tensor `[1, 1, N, 7]` where each of the N rows is `[batch, class_id, confidence, x1, y1, x2, y2]` in normalized 0-1 coordinates -- write a general parser for that shape.



In [ ]:
def parse_ssd_output(
    output: np.ndarray, image_shape: tuple, conf_threshold: float = 0.5
) -> list[dict]:
    """Parses the standard SSD-style [1, 1, N, 7] OpenCV DNN detection output format."""
    h, w = image_shape[:2]
    detections = []
    for i in range(output.shape[2]):
        confidence = float(output[0, 0, i, 2])
        if confidence < conf_threshold:
            continue
        class_id = int(output[0, 0, i, 1])
        x1, y1, x2, y2 = (output[0, 0, i, 3:7] * [w, h, w, h]).astype(int)
        detections.append(
            {"class_id": class_id, "confidence": confidence, "box": (x1, y1, x2, y2)}
        )
    return detections


# Parse the real network output from our face image
parsed = parse_ssd_output(output, image.shape, conf_threshold=0.5)
print("Parsed real detections (confidence >= 0.5 only):")
for det in parsed:
    print(f" - Class {det['class_id']} with {det['confidence'] * 100:.1f}% confidence at box {det['box']}")

## Part 2: Working with ONNX Models


### 1. Inspecting an ONNX model before use

Using the `onnx` library (not `cv2.dnn`) to read a model's declared input/output shapes and names -- this is the correct first step for any new ONNX model, before writing any preprocessing code.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, has_module


def inspect_onnx_model(path: str) -> dict:
    if not has_module("onnx"):
        print("onnx package not installed -- install with: pip install onnx")
        return {}
    import onnx

    model = onnx.load(path)
    info = {"inputs": [], "outputs": []}
    for inp in model.graph.input:
        dims = [
            d.dim_value if d.dim_value > 0 else "dynamic"
            for d in inp.type.tensor_type.shape.dim
        ]
        info["inputs"].append({"name": inp.name, "shape": dims})
    for out in model.graph.output:
        dims = [
            d.dim_value if d.dim_value > 0 else "dynamic"
            for d in out.type.tensor_type.shape.dim
        ]
        info["outputs"].append({"name": out.name, "shape": dims})
    return info


print("Usage (requires a real .onnx file on disk):")
print('  info = inspect_onnx_model("model.onnx")')
print(
    "This tells you the exact input shape/name to build cv2.dnn.blobFromImage calls around,"
)
print("instead of guessing based on a tutorial you found for a different model.")

### 2. Loading and running with cv2.dnn

Once shapes are known, loading is a single call -- the pattern mirrors the Caffe/TensorFlow loaders from notebook 34, since `cv2.dnn` normalizes the API across formats.


In [ ]:
def load_onnx_and_describe(path: str):
    net = cv2.dnn.readNetFromONNX(path)
    layer_names = net.getLayerNames()
    print(f"Loaded ONNX model with {len(layer_names)} layers")
    return net


print("Usage:")
print('  net = load_onnx_and_describe("model.onnx")')
print(
    "  blob = cv2.dnn.blobFromImage(image, scalefactor=1/255.0, size=(224, 224), swapRB=True)"
)
print("  net.setInput(blob)")
print("  output = net.forward()")
print()
print(
    "The scalefactor/size/swapRB values MUST match the model's documented training-time"
)
print(
    "preprocessing (often listed on the model's source repo/model card), not guessed."
)

### 3. A defensive inference wrapper

Wrap inference with an explicit shape check against the blob actually built, raising a clear error early rather than letting a shape mismatch fail deep inside `net.forward()` with a cryptic message.


In [ ]:
def safe_onnx_infer(net: "cv2.dnn.Net", blob: np.ndarray, expected_input_shape: tuple):
    if blob.shape != expected_input_shape:
        raise ValueError(
            f"Blob shape {blob.shape} does not match model's expected input shape "
            f"{expected_input_shape}. Check the model's documented preprocessing "
            f"(input size, channel order, normalization) before proceeding."
        )
    net.setInput(blob)
    return net.forward()


fake_blob = np.zeros((1, 3, 224, 224), dtype=np.float32)
try:
    safe_onnx_infer(net=None, blob=fake_blob, expected_input_shape=(1, 3, 416, 416))
except ValueError as e:
    print("Caught early, clear error instead of a cryptic OpenCV internal exception:")
    print(" ", e)

### 4. End-to-End ONNX Inference & Visualization

Here is a complete, working example where we load a real image, pass it to the YOLOv8 ONNX model using `cv2.dnn`, retrieve the raw output, decode the bounding boxes, apply Non-Maximum Suppression (NMS), and plot the final predictions using `matplotlib`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from cv_utils import load_real_image, get_real_data

# 1. Load YOLOv8 ONNX Model and COCO class names
class_names_path = get_real_data("models", "coco.names")
with open(class_names_path, "r") as f:
    class_names = [line.strip() for line in f.readlines()]

model_path = get_real_data("models", "yolov8n.onnx")
net = cv2.dnn.readNetFromONNX(model_path)

# 2. Load Real Image
img = load_real_image("images/landscapes", "city.jpg")

# 3. Preprocess and Run Inference
# YOLOv8 model expects 640x640 input resolution and 1/255.0 normalization
blob = cv2.dnn.blobFromImage(img, 1 / 255.0, (640, 640), swapRB=True, crop=False)
net.setInput(blob)
raw_output = net.forward()

# Transpose output from shape (1, 84, 8400) to (1, 8400, 84) to simplify parsing
raw_output = np.transpose(raw_output, (0, 2, 1))

# 4. Decode Detections (YOLOv8 custom decoding)
def decode_yolo_output(raw_output, image_shape, conf_threshold=0.4):
    h, w = image_shape[:2]
    x_factor = w / 640.0
    y_factor = h / 640.0
    
    detections = []
    for row in raw_output[0]:
        class_scores = row[4:]
        class_id = int(np.argmax(class_scores))
        confidence = float(class_scores[class_id])
        
        if confidence < conf_threshold:
            continue
            
        cx, cy, bw, bh = row[:4]
        x1 = int((cx - bw / 2) * x_factor)
        y1 = int((cy - bh / 2) * y_factor)
        box_w = int(bw * x_factor)
        box_h = int(bh * y_factor)
        
        detections.append({
            "class_id": class_id,
            "class_name": class_names[class_id],
            "confidence": confidence,
            "box": (x1, y1, box_w, box_h),
        })
    return detections

decoded = decode_yolo_output(raw_output, img.shape)

# 5. Apply Non-Maximum Suppression
boxes = [d["box"] for d in decoded]
scores = [d["confidence"] for d in decoded]
class_ids = [d["class_id"] for d in decoded]
indices = cv2.dnn.NMSBoxes(boxes, scores, 0.4, 0.4)

# 6. Draw Bounding Boxes and Plot Results
img_display = img.copy()
if len(indices) > 0:
    for i in indices.flatten():
        x, y, w, h = boxes[i]
        label = f"{class_names[class_ids[i]]}: {scores[i]:.2f}"
        cv2.rectangle(img_display, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(img_display, label, (x, max(y - 10, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('YOLOv8 ONNX Inference Results via cv2.dnn')
plt.show()


## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — OpenCV DNN Module Fundamentals: Sequential Multi-Model Pipeline inference

In production systems, deep learning models run sequentially to create pipelines. Here, we simulate a face-processing pipeline: first detection locates the boundary coordinate box, and then a classification model processes the crop.



In [ ]:
# Mock multi-model coordinator
class FaceEmotionPipeline:
    def __init__(self):
        print("Initialized Detection and Classification Models.")

    def run_inference(self, image: np.ndarray) -> tuple[list[int], str]:
        # Step 1: Detect face coordinate box
        # We mock detection returning (x, y, w, h)
        face_box = [50, 40, 100, 100]

        # Crop region of interest
        crop = image[40:140, 50:150]

        # Step 2: Feed crop to secondary classifier
        # We mock classification output returning a status label
        emotion = "Happy"

        return face_box, emotion


img = load_real_image("images/faces", "face.jpg")
pipeline = FaceEmotionPipeline()
box, label = pipeline.run_inference(img)
print(f"Pipeline output: Face box={box} | Classification result={label}")

### Mini Project — Working with ONNX Models: ONNX Model Parameter Quantization and Inference

ONNX models can be quantized (e.g. converting float32 parameters to float16) to speed up execution on supported GPUs. Here, we write a utility function that simulates ONNX model precision quantization.


In [ ]:
# Mock ONNX model parameter weights array
weights_fp32 = np.random.normal(0.0, 1.0, (100, 100)).astype(np.float32)


def quantize_weights_to_fp16(weights: np.ndarray) -> np.ndarray:
    """Quantize parameters to half-precision float16 format to reduce footprint."""
    weights_fp16 = weights.astype(np.float16)

    # Calculate file size reduction ratio
    size_32 = weights.nbytes
    size_16 = weights_fp16.nbytes
    print(
        f"FP32 size: {size_32} bytes | FP16 size: {size_16} bytes (Reduction: {size_32 / size_16:.1f}x)"
    )
    return weights_fp16


quantized = quantize_weights_to_fp16(weights_fp32)

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — OpenCV DNN Module Fundamentals
1. Download a real TensorFlow MobileNet-SSD model and run the full `readNetFromTensorFlow` -> `blobFromImage` -> `forward` -> `parse_ssd_output` pipeline end-to-end.
2. Extend `PREPROCESS_RECIPES` with the correct recipe for a PyTorch-exported ONNX classification model (typically ImageNet mean/std normalization).
3. Write `parse_ssd_output` variant for YOLO's different output tensor shape (covered in depth in notebook 35).

Use the empty cell below to work through them.



#### Solutions — OpenCV DNN Module Fundamentals

In [ ]:
# Solution 1: readNetFromTensorflow Caffe SSD model loading
proto_path = get_real_data("models", "opencv_face_detector.pbtxt")
model_path = get_real_data("models", "opencv_face_detector_uint8.pb")
net = cv2.dnn.readNetFromTensorflow(str(model_path), str(proto_path))

# Run end-to-end on face.jpg
img = load_real_image("images/faces", "face.jpg")
blob = cv2.dnn.blobFromImage(img, 1.0, (300, 300), (104.0, 177.0, 123.0), swapRB=False)
net.setInput(blob)
detections = net.forward()
results = parse_ssd_output(detections, img.shape, conf_threshold=0.5)
print("Exercise 1 Output (Real Face Detection):", results)

In [ ]:
# Solution 2: Preprocess recipe for PyTorch ONNX models
def preprocess_pytorch_onnx(image: np.ndarray) -> np.ndarray:
    """Apply standard ImageNet mean/std scaling used by PyTorch models."""
    # 1. Resize to target dimension
    resized = cv2.resize(image, (224, 224))
    # 2. Scale pixel values to [0.0, 1.0] range
    float_img = resized.astype(np.float32) / 255.0
    # 3. Standard ImageNet mean and standard deviation normalization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    normalized = (float_img - mean) / std
    # 4. Convert HWC to NCHW shape format
    blob = np.transpose(normalized, (2, 0, 1))
    return np.expand_dims(blob, axis=0)

In [ ]:
# Solution 3: SSD parser variant for YOLO shapes
def parse_yolo_outputs(net_outputs: np.ndarray, conf_thresh: float = 0.5) -> list[dict]:
    """Parse YOLO outputs (shape: [num_boxes, 85] where coords are x, y, w, h)."""
    detections = []
    for out in net_outputs:
        for detection in out:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > conf_thresh:
                # YOLO returns center_x, center_y, width, height
                cx, cy, w, h = detection[0:4]
                detections.append(
                    {
                        "bbox": (cx, cy, w, h),
                        "class_id": class_id,
                        "confidence": confidence,
                    }
                )
    return detections


# Test run
img = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
blob = preprocess_pytorch_onnx(img)
print("PyTorch ONNX blob shape:", blob.shape)

### Exercises — Working with ONNX Models
1. Export a small trained model from PyTorch (`torch.onnx.export`) or download a public one, and run it through `inspect_onnx_model` and `load_onnx_and_describe`.
2. Add output-shape validation (mirroring `safe_onnx_infer`'s input check) that flags unexpected output tensor shapes.
3. Write a benchmark comparing `cv2.dnn` ONNX inference speed against `onnxruntime` (a dedicated ONNX runtime) for the same model.

Use the empty cell below to work through them.


#### Solutions — Working with ONNX Models

In [ ]:
# Solution 1: PyTorch model ONNX export
# PyTorch export workflow:
# model = MyModel()
# dummy_input = torch.randn(1, 3, 224, 224)
# torch.onnx.export(model, dummy_input, "model.onnx", input_names=["input"], output_names=["output"])


In [ ]:
# Solution 2: Output shape validation check
def validate_onnx_output(output_tensor: np.ndarray, expected_shape: tuple) -> None:
    """Verify output dimensions match expectations before downstream usage."""
    if output_tensor.shape != expected_shape:
        raise ValueError(
            f"Dimension mismatch! Expected output shape: {expected_shape}, "
            f"but received shape: {output_tensor.shape}"
        )
    print("Output shape validation status: PASSED")

In [ ]:
# Solution 3: Benchmark cv2.dnn ONNX speed vs onnxruntime
# Explanation: `onnxruntime` is a dedicated engine with advanced layout optimization
# and custom kernel fusing, resulting in faster execution (1.5-2x) compared to
# OpenCV's general `cv2.dnn` reader.

import cv2
import numpy as np
import time
from cv_utils import get_real_data, has_module

# Get the path to a real model on disk
model_path = get_real_data("models", "yolov8n.onnx")

if not has_module("onnxruntime"):
    print("onnxruntime is not installed. Benchmark cannot be run.")
else:
    import onnxruntime as ort
    
    print(f"Benchmarking with model: {model_path}")
    
    # 1. Initialize cv2.dnn network
    net = cv2.dnn.readNetFromONNX(model_path)
    
    # 2. Initialize onnxruntime session
    session = ort.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name
    
    # Create a dummy input blob matching YOLOv8 (1, 3, 640, 640)
    blob = np.random.randn(1, 3, 640, 640).astype(np.float32)
    
    # Warmup runs to initialize engine internal structures
    net.setInput(blob)
    _ = net.forward()
    _ = session.run(None, {input_name: blob})
    
    # Measure cv2.dnn latency
    num_runs = 20
    start_time = time.perf_counter()
    for _ in range(num_runs):
        net.setInput(blob)
        _ = net.forward()
    dnn_latency = (time.perf_counter() - start_time) / num_runs
    
    # Measure onnxruntime latency
    start_time = time.perf_counter()
    for _ in range(num_runs):
        _ = session.run(None, {input_name: blob})
    ort_latency = (time.perf_counter() - start_time) / num_runs
    
    print(f"cv2.dnn average latency: {dnn_latency * 1000:.2f} ms")
    print(f"onnxruntime average latency: {ort_latency * 1000:.2f} ms")
    print(f"onnxruntime Speedup over cv2.dnn: {dnn_latency / ort_latency:.2f}x")


## Summary

You can load an ONNX model defensively, reproduce its preprocessing contract, inspect output tensors, and build a reusable inference wrapper.

- **Best Practices:** Version model files, record preprocessing and labels with the model, validate tensor shapes, and test a known input before optimizing performance.
- **Common Pitfalls:** Guessing resize/channel/normalization values, assuming every ONNX export is compatible, and hard-coding an output tensor layout.